# Legal & Compliance Text Summarization with BART (fine-tuned)
**Synthra AI take-home assessment, Ref SA-NLP-2026-014**

This notebook runs top to bottom on a free Colab GPU (**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**).

### Approach in one paragraph
An **abstractive** Transformer **encoder–decoder** (`facebook/bart-large-cnn`, a pretrained BART checkpoint) is fine-tuned on legal text (BillSum) using **transfer learning**. Documents longer than the model's 1,024-token window are handled by sentence-aware **chunking → per-chunk summarisation → merge/dedupe (→ second pass if still long)**. Because legal text is high-stakes, every summary also goes through a **faithfulness check** that flags numbers, dates and capitalised terms that do not appear in the source.

### How this notebook maps to the brief
| Brief section | Where in this notebook |
|---|---|
| 4.1 Input & length support, compression ratio | Sections 3 (ratio analysis), 5 (chunking), 8 (pipeline) |
| 4.2 Self-attention, multi-head attention, positional encoding, transfer learning | Section 6 (architecture walkthrough), Section 10 (fine-tuning) |
| 4.3 Processing pipeline (preprocess → tokenise → summarise → post-process) | Sections 4, 5, 8 |
| 4.4 Faithfulness & intent preservation | Section 7 (faithfulness checker), Section 12 (key-fact retention) |
| 4.5 Quality (ROUGE-1/2/L, fluency) | Sections 9, 11 |
| 4.6 Generalisation | Section 11 (unseen California split), Section 12 (hand-written contracts) |
| 7. Dry run for the video | Section 13 |

### Disclosure of third-party components (as the brief requires)
Hugging Face `transformers` (model, tokenizer, `Seq2SeqTrainer`), `datasets`, `accelerate`, PyTorch, Google's `rouge_score`, `facebook/bart-large-cnn` (pretrained checkpoint), and the BillSum dataset (`FiscalNote/billsum`). The chunking, cleaning, deduplication, faithfulness checker and evaluation code are written for this notebook.

## 1. Install the latest libraries
`-U` upgrades to the newest releases. This notebook was written against **transformers 5.x** and **datasets 5.x**. Important API changes it already accounts for: `eval_strategy` (not `evaluation_strategy`), `processing_class` (not `tokenizer`) in the Trainer, and no `summarization` pipeline task, so generation is done with `model.generate` directly.

In [1]:
# Install / upgrade everything this notebook needs (takes about a minute).
!pip install -q -U transformers datasets accelerate rouge_score sentencepiece matplotlib

^C


## 2. Imports and configuration
All tunable settings live in one cell. The defaults are sized for a free T4 GPU (roughly 30 to 60 minutes in total, depending on GPU availability). If you hit a CUDA out-of-memory error, set `TRAIN_BATCH = 1` and `GRAD_ACCUM = 16`. Lower `N_TRAIN` or `NUM_EPOCHS` for a quicker run. Set `DO_FINE_TUNE = False` to skip training and use the pretrained model as-is.

In [ ]:
import os, re, math, copy, random, json, time, shutil, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"          # silences tokenizer fork warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import torch
import matplotlib.pyplot as plt
import transformers, datasets
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, set_seed)
from datasets import load_dataset
from rouge_score import rouge_scorer

transformers.logging.set_verbosity_error()          # hide repetitive generation-length notices (we set lengths explicitly)
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)

# ---------------- configuration ----------------
SEED            = 42
MODEL_NAME      = "facebook/bart-large-cnn"   # pretrained encoder-decoder checkpoint (transfer learning)
DATASET_NAME    = "FiscalNote/billsum"        # public legal summarisation dataset (US bills)
DO_FINE_TUNE    = True                        # False = use the pretrained model without training

MAX_SOURCE_TOKENS = 1024     # BART's encoder window (learned positions go up to 1024)
MAX_TARGET_TOKENS = 256      # summary length used for training targets
N_TRAIN, N_VAL, N_TEST, N_UNSEEN = 2000, 150, 100, 60   # subset sizes (T4-friendly)

NUM_EPOCHS      = 2
LEARNING_RATE   = 3e-5
TRAIN_BATCH     = 2          # per-step batch size (small, to fit a 16 GB T4)
GRAD_ACCUM      = 8          # effective batch size = TRAIN_BATCH * GRAD_ACCUM = 16
EVAL_BATCH      = 4

# ---------------- summarisation pipeline settings ----------------
TARGET_COMPRESSION = 0.20    # summary is about 20% of the source length (justified in Section 3)
CHUNK_TOKENS       = 900     # tokens per chunk (below 1024 to leave room for special tokens)
FINAL_CAP_TOKENS   = 350     # long documents are capped here (about 260 words)
MAX_LEVELS         = 3       # max rounds of "summarise the summaries"

OUTPUT_DIR, FINAL_DIR = "bart_legal_ckpt", "bart_legal_final"

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU - switch the runtime to T4 GPU")

## 3. Load the dataset and justify the compression ratio
**BillSum** (US Congressional bills with expert-written summaries) is the largest public legal-summarisation dataset that loads reliably. It is *statutory* text rather than contracts, so Section 12 also tests on hand-written contracts and notices the model never saw.

Splits used: `train` (training + validation), `test` (in-domain evaluation), and `ca_test` (California bills, a different drafting style, which serves as the **unseen** generalisation set).

In [ ]:
raw = load_dataset(DATASET_NAME)
print(raw)

# Keep examples that have real content, shuffle with a fixed seed, then carve out the subsets.
def keep(ex):
    return len(ex["text"]) >= 500 and len(ex["summary"]) >= 50

def take(ds, start, n):
    end = min(start + n, len(ds))
    return ds.select(range(start, end))

train_full = raw["train"].filter(keep).shuffle(seed=SEED)
train_ds  = take(train_full, 0, N_TRAIN)
val_ds    = take(train_full, N_TRAIN, N_VAL)
test_ds   = take(raw["test"].filter(keep).shuffle(seed=SEED), 0, N_TEST)
unseen_ds = take(raw["ca_test"].filter(keep).shuffle(seed=SEED), 0, N_UNSEEN)
print({"train": len(train_ds), "val": len(val_ds), "test": len(test_ds), "unseen(ca_test)": len(unseen_ds)})

# ---- data-driven compression-ratio justification (brief section 4.1 / 4.5) ----
wc = lambda s: len(s.split())
sample = take(train_ds, 0, 500)
ratios = [wc(e["summary"]) / wc(e["text"]) for e in sample]
print(f"\nAvg source words  : {np.mean([wc(e['text']) for e in sample]):.0f}")
print(f"Avg summary words : {np.mean([wc(e['summary']) for e in sample]):.0f}")
print(f"Summary/source ratio  median={np.median(ratios):.2f}  mean={np.mean(ratios):.2f}")
print(f"-> TARGET_COMPRESSION = {TARGET_COMPRESSION} (about {1/TARGET_COMPRESSION:.0f}:1). Long documents are additionally capped at {FINAL_CAP_TOKENS} tokens.")

**Compression-ratio decision.** Expert summaries in the data are roughly a fifth to a third of the source length. I use a target of **about 20% (5:1)**. It is short enough to be a real reduction, yet long enough to keep parties, dates, amounts and obligations. For very long contracts a fixed 20% would still be several pages, so the output is capped at about 350 tokens (about 260 words), which means the effective ratio drops to roughly 5 to 10% on 10+ page documents. For tiny inputs such as a single clause, a floor on summary length avoids producing a meaningless one-liner.

## 4. Preprocessing and sentence splitting
Legal text has quirks that break naive processing: abbreviations such as `Sec.`, `No.`, `Pvt. Ltd.`, `U.S.C.` that fool sentence splitters, LaTeX-style quote marks in BillSum (` `` ` and `''`), and irregular line breaks. `clean_text` normalises whitespace and quotes without touching any facts. `split_sentences` is a rule-based splitter protected against those abbreviations.

In [ ]:
def clean_text(t):
    t = t.replace("\r", "\n")
    t = t.replace("``", '"').replace("''", '"')      # BillSum uses LaTeX-style quotes
    t = re.sub(r"_{2,}", " ", t)                      # underscore rules/blank lines
    t = re.sub(r"\s+", " ", t)                        # collapse all whitespace (incl. newlines, nbsp)
    return t.strip()

# Abbreviations whose trailing "." must NOT end a sentence.
_ABBR = ["Sec", "Secs", "No", "Nos", "Inc", "Ltd", "Pvt", "Co", "Corp", "LLC", "LLP", "Art", "Arts", "Cl",
         "Para", "Fig", "Ch", "Pt", "Reg", "Regs", "Subsec", "Dept", "St", "Mr", "Mrs", "Ms", "Dr", "Jr", "Sr",
         "Hon", "Gen", "Gov", "Sen", "Rep", "Prof", "Rev", "Stat", "Cir", "vs", "v", "approx", "Rs", "cf", "al",
         "etc", "Jan", "Feb", "Mar", "Apr", "Jun", "Jul", "Aug", "Sep", "Sept", "Oct", "Nov", "Dec"]
_ABBR_RE  = re.compile(r"\b(" + "|".join(sorted(map(re.escape, _ABBR), key=len, reverse=True)) + r")\.")
_MULTI_RE = re.compile(r"\b(U\.S\.C|U\.S\.A|U\.S|U\.K|e\.g|i\.e|a\.m|p\.m)\.")
# A boundary = . ! ? (optionally followed by a closing quote/bracket), whitespace, then an uppercase letter/digit/opener.
_SPLIT_RE = re.compile(r"""(?:(?<=[.!?])|(?<=[.!?]["')\]]))\s+(?=["'(\[]?[A-Z0-9])""")

# A bare clause number such as "2." at the start of a sentence ("... remedy. 2. Payment is due.") is a list marker, not a sentence end.
_LISTNUM_RE = re.compile(r"""(^|[.!?]["')\]]?\s+)(\d{1,2})\.(?=\s)""")

def split_sentences(text):
    t = _MULTI_RE.sub(lambda m: m.group(0)[:-1] + "<DOT>", text)   # mask protected dots
    t = _ABBR_RE.sub(lambda m: m.group(1) + "<DOT>", t)
    t = _LISTNUM_RE.sub(lambda m: m.group(1) + m.group(2) + "<DOT>", t)
    parts = _SPLIT_RE.split(t)
    return [p.replace("<DOT>", ".").strip() for p in parts if p.strip()]

demo = 'Under Sec. 4 of the Act, Acme Pvt. Ltd. shall pay Rs. 5,000 (see 42 U.S.C. 1396). The fee is 1.5% per month. "Term" means two years. 2. Payment is due.'
for i, s in enumerate(split_sentences(clean_text(demo)), 1):
    print(i, "|", s)

## 5. Chunking for long documents
BART reads at most 1,024 tokens, but a master agreement can be many times longer. The chunker:
1. splits the document into sentences (so chunks never cut a sentence in half),
2. packs sentences greedily until a chunk reaches `CHUNK_TOKENS`,
3. repeats the last sentence of each chunk at the start of the next one (**overlap**) so context is not lost at the boundary,
4. hard-splits any single sentence longer than a chunk.

In [ ]:
def chunk_document(text, tokenizer, max_tokens=CHUNK_TOKENS, overlap_sentences=1):
    sents = split_sentences(text)
    if not sents:
        return [text]
    ids = tokenizer(sents, add_special_tokens=False)["input_ids"]      # token counts per sentence

    # 1) break any single over-long sentence into token windows
    pieces = []                                                         # (text, n_tokens)
    for s, tok_ids in zip(sents, ids):
        if len(tok_ids) <= max_tokens:
            pieces.append((s, len(tok_ids)))
        else:
            for i in range(0, len(tok_ids), max_tokens):
                part = tok_ids[i:i + max_tokens]
                pieces.append((tokenizer.decode(part), len(part)))

    # 2) greedy packing with sentence overlap
    chunks, cur, cur_len = [], [], 0
    for s, n in pieces:
        if cur and cur_len + n > max_tokens:
            chunks.append(" ".join(p for p, _ in cur))
            carry = cur[-overlap_sentences:] if overlap_sentences > 0 else []
            if sum(m for _, m in carry) + n > max_tokens:
                carry = []
            cur = list(carry)
            cur_len = sum(m for _, m in cur)
        cur.append((s, n))
        cur_len += n
    if cur:
        chunks.append(" ".join(p for p, _ in cur))
    return chunks

## 6. Load the pretrained model and walk through the architecture
`facebook/bart-large-cnn` is a **Transformer encoder–decoder**: a bidirectional encoder reads the document, and an autoregressive decoder writes the summary while *cross-attending* to the encoder output. It was pretrained as a denoising autoencoder and then tuned for summarisation; we **transfer** that knowledge and fine-tune it on legal text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

cfg = model.config
print(f"Parameters           : {sum(p.numel() for p in model.parameters())/1e6:.0f} M")
print(f"Encoder / decoder    : {cfg.encoder_layers} layers / {cfg.decoder_layers} layers")
print(f"Hidden size (d_model): {cfg.d_model}")
print(f"Attention heads      : {cfg.encoder_attention_heads} (encoder), {cfg.decoder_attention_heads} (decoder)  -> head dim = {cfg.d_model // cfg.encoder_attention_heads}")
print(f"Max positions        : {cfg.max_position_embeddings}   | vocab size: {cfg.vocab_size}")
print("Positional module    :", type(model.model.encoder.embed_positions).__name__)
print("Pretrained generation defaults:", {k: getattr(model.generation_config, k, None) for k in ["num_beams", "max_length", "min_length", "no_repeat_ngram_size", "length_penalty"]})

### 6.1 Seeing the mechanisms the brief asks about
* **Self-attention** lets each token build its representation from every other token in the same sequence (e.g. linking "the Provider" to its defined meaning many sentences earlier).
* **Multi-head attention** runs several attention operations in parallel in different learned subspaces (BART-large: 16 heads of size 64), so one head can track party names while another tracks dates or obligations.
* **Positional encoding**: attention itself has no notion of order, so position information is added to the token embeddings. *BART uses learned positional embeddings* (a trainable table of 1,024 positions) rather than the fixed sinusoids of the original Transformer. The role is identical.
* **Cross-attention** (decoder → encoder) is how each generated summary word looks back at the source.

The next cell proves each point on a real sentence. It uses a CPU copy with the `eager` attention implementation, because the faster kernels do not return attention weights.

In [ ]:
demo_text = "This Agreement is effective as of 1 July 2026 between Acme Ltd and Beta Pvt Ltd."
demo_summary = "Acme and Beta agree from 1 July 2026."

# CPU copy with eager attention so that attention weights can be returned (the training model is untouched).
probe = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, attn_implementation="eager").eval()
enc = tokenizer(demo_text, return_tensors="pt")
lab = tokenizer(text_target=demo_summary, return_tensors="pt")
with torch.no_grad():
    out = probe(**enc, labels=lab["input_ids"], output_attentions=True)

print("Tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("\n[Self-attention + Multi-head] encoder attention, layer 12:", tuple(out.encoder_attentions[-1].shape), "= (batch, heads, query_tokens, key_tokens)")
print("[Cross-attention] decoder->encoder, layer 12            :", tuple(out.cross_attentions[-1].shape), "= (batch, heads, summary_tokens, source_tokens)")

# ---- positional encoding demo: same word, different positions ----
text2 = "the Party shall pay and the Party shall deliver"
ids2 = tokenizer(text2, return_tensors="pt", add_special_tokens=False)["input_ids"]
toks2 = tokenizer.convert_ids_to_tokens(ids2[0])
enc_mod = probe.model.encoder
with torch.no_grad():
    tok_emb = enc_mod.embed_tokens(ids2)[0]                       # word embedding only
    pos_emb = enc_mod.embed_positions(ids2)                       # learned position embedding
    pos_emb = pos_emb[0] if pos_emb.dim() == 3 else pos_emb
    with_pos = tok_emb + pos_emb
cos = torch.nn.functional.cosine_similarity
i1, i2 = [i for i, t in enumerate(toks2) if "Party" in t][:2]
print(f"\n[Positional encoding] the two occurrences of '{toks2[i1]}' (positions {i1} and {i2}):")
print(f"  cosine similarity, word embeddings only   : {cos(tok_emb[i1], tok_emb[i2], dim=0).item():.4f}  (identical -> order is invisible)")
print(f"  cosine similarity, word + position embeds : {cos(with_pos[i1], with_pos[i2], dim=0).item():.4f}  (now distinguishable by position)")
print("  learned position table shape:", tuple(enc_mod.embed_positions.weight.shape))

# ---- visualise attention for the last encoder layer ----
labels = [t.replace("Ġ", "") for t in tokenizer.convert_ids_to_tokens(enc["input_ids"][0])]
att = out.encoder_attentions[-1][0]                                # (heads, L, L)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in zip(axes, [att[0], att.mean(0)], ["Head 0", "Average of all 16 heads"]):
    ax.imshow(mat.numpy(), cmap="viridis")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(f"Encoder self-attention, last layer - {title}")
plt.tight_layout(); plt.show()

del probe, out

## 7. Metrics: ROUGE and a faithfulness checker
**ROUGE-1/2/L** (F-measure, stemmed) measure word, bigram and longest-common-subsequence overlap with the expert summary.

ROUGE cannot tell whether a *number* was altered, and in legal text that is the worst possible error. So there is also a lightweight **faithfulness checker** that flags anything in the summary that is not in the source:
* **numbers** (amounts, percentages, day counts, years),
* **dates** (parsed so that "1 July 2026" equals "July 1, 2026"),
* **capitalised terms** (party names, defined terms, places).

It is a heuristic safety net (it cannot catch a *wrong* but present number, or a reversed obligation), so the output also lists warnings for a human to review rather than silently editing the text.

In [ ]:
# ----------------------------- ROUGE -----------------------------
_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def rouge_scores(preds, refs):
    per = [_scorer.score(r, p) for p, r in zip(preds, refs)]                     # score(target, prediction)
    return {k: 100 * float(np.mean([s[k].fmeasure for s in per])) for k in ["rouge1", "rouge2", "rougeL"]}

# --------------------- faithfulness checker ----------------------
_MONTHS = {m: i + 1 for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}
_MON = r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|June?|July?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
_DATE_MDY = re.compile(rf"\b({_MON})\.?\s+(\d{{1,2}})(?:st|nd|rd|th)?,?\s+(\d{{4}})\b")
_DATE_DMY = re.compile(rf"\b(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:day\s+of\s+)?({_MON})\.?,?\s+(\d{{4}})\b")
_DATE_ISO = re.compile(r"\b(\d{4})-(\d{2})-(\d{2})\b")
_NUM_RE   = re.compile(r"\d[\d,]*(?:\.\d+)?")
_WORD_RE  = re.compile(r"[A-Za-z][A-Za-z'&\-]*")

def extract_dates(text):
    found = set()
    for m, d, y in _DATE_MDY.findall(text):
        found.add((int(y), _MONTHS[m[:3].lower()], int(d)))
    for d, m, y in _DATE_DMY.findall(text):
        found.add((int(y), _MONTHS[m[:3].lower()], int(d)))
    for y, m, d in _DATE_ISO.findall(text):
        found.add((int(y), int(m), int(d)))
    return found

def extract_numbers(text):
    return {n.replace(",", "") for n in _NUM_RE.findall(text)}

def _norm_word(w):
    w = re.sub(r"'s$", "", w.lower())
    return w.rstrip("s")                                  # crude plural folding (Party / Parties-> parti, ok for matching)

def proper_terms(text):
    # capitalised words that are not simply the first word of a sentence, plus ALL-CAPS words
    out = set()
    for sent in split_sentences(text):
        for i, w in enumerate(_WORD_RE.findall(sent)):
            if len(w) >= 2 and (w.isupper() or (w[0].isupper() and i > 0)):
                out.add(w)
    return out

def faithfulness_report(source, summary):
    src_nums, sum_nums = extract_numbers(source), extract_numbers(summary)
    bad_nums  = sorted(sum_nums - src_nums)
    bad_dates = sorted(extract_dates(summary) - extract_dates(source))
    src_words = {_norm_word(w) for w in _WORD_RE.findall(source)}
    sum_terms = proper_terms(summary)
    bad_terms = sorted(w for w in sum_terms if _norm_word(w) not in src_words)
    return {"unsupported_numbers": bad_nums, "unsupported_dates": ["%04d-%02d-%02d" % d for d in bad_dates],
            "unsupported_terms": bad_terms, "n_numbers": len(sum_nums), "n_terms": len(sum_terms)}

def fact_present(fact, summary):
    # a key fact counts as retained if it appears verbatim (ignoring commas/case) or, for dates, as the same calendar date
    d = extract_dates(fact)
    if d:
        return d <= extract_dates(summary)
    norm = lambda s: re.sub(r"[\s,]+", " ", s.lower()).strip()
    return norm(fact) in norm(summary)

# quick self-test of the checker
_src = "The Provider shall pay USD 240,000 by 15 January 2026 to Ardent Logistics Limited."
print(faithfulness_report(_src, "Ardent Logistics must pay USD 240,000 by January 15, 2026."))   # nothing unsupported
print(faithfulness_report(_src, "Ardent Logistics must pay USD 250,000 by 16 January 2026 to Zenith Corp."))  # flags amount, date, Zenith

## 8. Generation utilities and the end-to-end summarisation pipeline
`summarize_document(text)` is the single entry point the brief asks for (4.3):

**raw text → clean → chunk → summarise each chunk → merge + remove duplicate sentences → (summarise again if still long) → tidy → faithfulness check.**

Each chunk gets its own length budget derived from `TARGET_COMPRESSION` (with a floor and cap), and we explicitly override the pretrained model's built-in `min_length=56`, which would otherwise force padding on short inputs.

In [ ]:
BASE_GEN = dict(num_beams=4, length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True, min_length=0)

def run_generation(texts, model, tokenizer, gen_kwargs, batch_size=4, max_source_tokens=MAX_SOURCE_TOKENS):
    # Batched beam-search generation; longest inputs first so batches need little padding.
    model.eval()
    order = np.argsort([-len(t) for t in texts])
    results = [None] * len(texts)
    use_amp = model.device.type == "cuda"
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        enc = tokenizer(batch, max_length=max_source_tokens, truncation=True, padding=True, return_tensors="pt").to(model.device)
        with torch.inference_mode(), torch.autocast(device_type=model.device.type, dtype=torch.float16, enabled=use_amp):
            out = model.generate(**enc, **gen_kwargs)
        for i, text in zip(idx, tokenizer.batch_decode(out, skip_special_tokens=True)):
            results[i] = text.strip()
    return results

def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def length_budget(n_in):
    # Summary length for a piece of n_in tokens: ~TARGET_COMPRESSION of it, with a floor (tiny clauses) and a cap.
    target = int(round(TARGET_COMPRESSION * n_in))
    max_new = int(min(max(target * 1.25, 24), 200))
    min_new = int(min(max(target * 0.5, 8), max_new - 8))
    return min_new, max_new

def finalize_summary(text):
    # Post-processing: drop near-duplicate sentences (from chunk overlap) and a dangling half-finished last sentence.
    kept, kept_sets = [], []
    for s in split_sentences(text):
        toks = set(re.findall(r"\w+", s.lower()))
        if not toks:
            continue
        if any(len(toks & k) / max(1, len(toks | k)) >= 0.8 for k in kept_sets):
            continue
        kept.append(s); kept_sets.append(toks)
    out = " ".join(kept)
    if out and out[-1] not in ".!?\"')" and len(kept) > 1:
        out = " ".join(kept[:-1])
    return out.strip()

def trim_to_tokens(text, max_tokens):
    out = []
    for s in split_sentences(text):
        if n_tokens(" ".join(out + [s])) > max_tokens:
            break
        out.append(s)
    return " ".join(out) if out else text

def summarize_document(text, model, tokenizer, chunk_tokens=CHUNK_TOKENS, overlap_sentences=1, num_beams=4):
    source = clean_text(text)
    src_tok = n_tokens(source)
    level_text, trace = source, []
    for level in range(MAX_LEVELS):
        chunks = chunk_document(level_text, tokenizer, chunk_tokens, overlap_sentences if level == 0 else 0)
        pieces = []
        for c in chunks:
            mn, mx = length_budget(n_tokens(c))
            gen = dict(BASE_GEN, num_beams=num_beams, min_new_tokens=mn, max_new_tokens=mx)
            pieces.append(run_generation([c], model, tokenizer, gen, batch_size=1)[0])
        merged = finalize_summary(" ".join(pieces))
        trace.append({"level": level + 1, "chunks": len(chunks), "summary_tokens": n_tokens(merged)})
        if len(chunks) == 1 or n_tokens(merged) <= FINAL_CAP_TOKENS:
            break
        level_text = merged                                   # still too long: summarise the summaries
    if n_tokens(merged) > FINAL_CAP_TOKENS:
        merged = trim_to_tokens(merged, FINAL_CAP_TOKENS)
    report = faithfulness_report(source, merged)
    warnings_ = ([f"number not found in source: {n}" for n in report["unsupported_numbers"]]
                 + [f"date not found in source: {d}" for d in report["unsupported_dates"]]
                 + [f"term not found in source: {t}" for t in report["unsupported_terms"]])
    s_tok = n_tokens(merged)
    return {"summary": merged, "source_tokens": src_tok, "summary_tokens": s_tok,
            "compression_ratio": s_tok / max(1, src_tok), "levels": trace, "warnings": warnings_}

## 9. Baseline: the pretrained model *before* fine-tuning
We score the untouched checkpoint first so the fine-tuning gain can be measured honestly. For a fair comparison, the baseline and the fine-tuned model use exactly the same decoding settings on exactly the same documents (single pass over the first 1,024 tokens, which is also how training works).

In [ ]:
EVAL_GEN = dict(BASE_GEN, max_new_tokens=256, min_new_tokens=64)
RESULTS = {}

def evaluate_split(model, tokenizer, ds, label):
    sources = [clean_text(t) for t in ds["text"]]
    refs    = [clean_text(s) for s in ds["summary"]]
    t0 = time.time()
    preds = run_generation(sources, model, tokenizer, EVAL_GEN, batch_size=EVAL_BATCH)
    secs = time.time() - t0
    scores = rouge_scores(preds, refs)
    reps = [faithfulness_report(s, p) for s, p in zip(sources, preds)]
    tot_n = max(1, sum(r["n_numbers"] for r in reps))
    tot_t = max(1, sum(r["n_terms"] for r in reps))
    res = {**scores,
           "avg_summary_words": float(np.mean([len(p.split()) for p in preds])),
           "avg_ref_words": float(np.mean([len(r.split()) for r in refs])),
           "unsupported_number_rate_%": 100 * sum(len(r["unsupported_numbers"]) for r in reps) / tot_n,
           "unsupported_term_rate_%": 100 * sum(len(r["unsupported_terms"]) for r in reps) / tot_t,
           "docs_with_unsupported_number_%": 100 * float(np.mean([len(r["unsupported_numbers"]) > 0 for r in reps])),
           "n_docs": len(preds), "gen_seconds": secs}
    RESULTS[label] = res
    print(f"\n=== {label} ===")
    for k, v in res.items():
        print(f"  {k:32s}: {v:8.2f}")
    return preds

baseline_test_preds   = evaluate_split(model, tokenizer, test_ds,   "pretrained | BillSum test")
baseline_unseen_preds = evaluate_split(model, tokenizer, unseen_ds, "pretrained | California (unseen)")
if device == "cuda":
    torch.cuda.empty_cache()

## 10. Fine-tuning (transfer learning)
**Why fine-tune?** Section 4.2 of the brief explicitly lists transfer learning with a pretrained encoder–decoder "fine-tuned for the summarization task". The stock checkpoint was tuned on news, whose summaries have a different style and emphasis from legal text.

**Memory/speed choices for a 16 GB T4:** batch size 2 with gradient accumulation 8 (effective batch 16), gradient checkpointing, mixed precision (fp16 on T4, bf16 on newer GPUs), checkpoints that store only the model weights. Label smoothing 0.1 is a common regulariser for sequence-to-sequence fine-tuning.

**Tokenisation:** the *source* is tokenised with BART's byte-level BPE (truncated at 1,024 tokens). The *summary* is tokenised as `text_target`, which becomes the `labels`. The Trainer shifts those labels right to form the decoder input, which is **teacher forcing**.

In [ ]:
def tokenize_batch(batch):
    # Source: cleaned text -> BPE token ids (truncate to the encoder window)
    inputs  = tokenizer([clean_text(t) for t in batch["text"]], max_length=MAX_SOURCE_TOKENS, truncation=True)
    # Target: cleaned reference summary -> token ids used as labels
    targets = tokenizer(text_target=[clean_text(s) for s in batch["summary"]], max_length=MAX_TARGET_TOKENS, truncation=True)
    inputs["labels"] = targets["input_ids"]
    return inputs

if DO_FINE_TUNE:
    tok_train = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names, desc="tokenising train")
    tok_val   = val_ds.map(tokenize_batch,   batched=True, remove_columns=val_ds.column_names,   desc="tokenising val")
    print(tok_train)
    ex = tok_train[0]
    print("example input ids :", ex["input_ids"][:12], "...", f"({len(ex['input_ids'])} tokens)")
    print("example label ids :", ex["labels"][:12], "...", f"({len(ex['labels'])} tokens)")

In [ ]:
if DO_FINE_TUNE:
    # Pads each batch dynamically and sets padded label positions to -100 so they are ignored by the loss.
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

    steps_per_epoch = math.ceil(len(tok_train) / (TRAIN_BATCH * GRAD_ACCUM))
    total_steps     = steps_per_epoch * NUM_EPOCHS
    warmup_steps    = max(1, int(0.05 * total_steps))                 # 5% linear warm-up
    eval_every      = max(5, steps_per_epoch // 2)                    # evaluate twice per epoch
    use_bf16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # T4 (cc 7.5) -> fp16 instead

    args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        label_smoothing_factor=0.1,
        gradient_checkpointing=True,
        fp16=(device == "cuda" and not use_bf16),
        bf16=use_bf16,
        eval_strategy="steps", eval_steps=eval_every,
        save_strategy="steps", save_steps=eval_every, save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=10,
        predict_with_generate=False,         # validation loss only during training; ROUGE is computed afterwards
        report_to="none",
        seed=SEED,
    )

    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tok_train, eval_dataset=tok_val,
                             data_collator=collator, processing_class=tokenizer)
    print(f"{len(tok_train)} train examples | {total_steps} optimisation steps | evaluating every {eval_every} steps")
    train_out = trainer.train()
    print("training finished:", {k: round(v, 4) for k, v in train_out.metrics.items() if isinstance(v, float)})
else:
    print("DO_FINE_TUNE = False -> skipping training; the pretrained model is used as-is.")

In [ ]:
if DO_FINE_TUNE:
    # Training / validation loss curves
    log = trainer.state.log_history
    tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
    ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
    plt.figure(figsize=(7, 4))
    if tr: plt.plot(*zip(*tr), label="train loss")
    if ev: plt.plot(*zip(*ev), marker="o", label="validation loss")
    plt.xlabel("optimisation step"); plt.ylabel("loss (with label smoothing)"); plt.legend(); plt.grid(alpha=.3)
    plt.title("Fine-tuning curves"); plt.show()

    model.config.use_cache = True                  # re-enable the decoding cache (it was off for gradient checkpointing)
    model.eval()
    trainer.save_model(FINAL_DIR)                  # save the best checkpoint
    tokenizer.save_pretrained(FINAL_DIR)
    print("saved to", FINAL_DIR)

## 11. Evaluation after fine-tuning
Same documents, same decoding settings as the baseline. The "California (unseen)" split has a different drafting style from the training data (generalisation, brief section 4.6).

In [ ]:
label = "fine-tuned" if DO_FINE_TUNE else "pretrained (no FT, repeat)"
ft_test_preds   = evaluate_split(model, tokenizer, test_ds,   f"{label} | BillSum test")
ft_unseen_preds = evaluate_split(model, tokenizer, unseen_ds, f"{label} | California (unseen)")

# ---- side-by-side table + chart ----
keys  = ["rouge1", "rouge2", "rougeL", "unsupported_number_rate_%", "unsupported_term_rate_%"]
heads = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "bad num %", "bad term %"]      # bad = not found in the source document
print("\n%-38s" % "run" + "".join("%12s" % h for h in heads))
for name, r in RESULTS.items():
    print("%-38s" % name + "".join("%12.2f" % r[k] for k in keys))

names = list(RESULTS)
x = np.arange(3); w = 0.2
plt.figure(figsize=(8, 4))
for i, n in enumerate(names):
    plt.bar(x + i * w, [RESULTS[n][k] for k in ["rouge1", "rouge2", "rougeL"]], w, label=n)
plt.xticks(x + w * (len(names) - 1) / 2, ["ROUGE-1", "ROUGE-2", "ROUGE-L"]); plt.ylabel("F1 x 100")
plt.legend(fontsize=7); plt.title("ROUGE: before vs after fine-tuning"); plt.grid(axis="y", alpha=.3); plt.show()

# ---- one qualitative example ----
i = 0
print("\nREFERENCE (expert):\n", clean_text(test_ds[i]["summary"])[:700])
print("\nMODEL SUMMARY:\n", ft_test_preds[i][:700])

**Reading the results.** Look at three things together: (1) the ROUGE gain after fine-tuning, (2) the generated summary length versus the reference length, and (3) the *unsupported number / term rates*, which are the hallucination proxies. ROUGE alone is not enough for legal text.

## 12. Unseen documents: contracts, an NDA, a short notice and a regulatory notice
These four documents were written for this notebook (all parties and the regulator are **fictional**). They are not in BillSum and span different drafting styles and lengths, from a single clause to a multi-section agreement. There are no reference summaries, so quality is judged by (a) reading the output, (b) the faithfulness warnings, and (c) **key-fact retention**: whether the parties, dates and amounts we marked as essential survive in the summary.

In [ ]:
SAMPLES = {
"Short notice (single clause)": dict(
text="""Notice of Termination. Pursuant to Clause 12.2 of the Supply Agreement dated 14 February 2025 between Harbor Textiles Pvt. Ltd. (the "Supplier") and Lumen Retail Private Limited (the "Buyer"), the Buyer hereby gives the Supplier sixty (60) days' written notice of termination. The termination shall take effect on 30 September 2026, and all outstanding invoices, including the sum of INR 18,75,000, shall be settled on or before that date.""",
facts=["Harbor Textiles", "Lumen Retail", "30 September 2026", "18,75,000"]),

"Mutual NDA": dict(
text="""MUTUAL NON-DISCLOSURE AGREEMENT. This Mutual Non-Disclosure Agreement (the "Agreement") is entered into as of 1 March 2026 (the "Effective Date") between Northwind Analytics Inc., a Delaware corporation ("Northwind"), and Kestrel Biosciences GmbH, a company organised under the laws of Germany ("Kestrel"). 1. Purpose. The parties wish to explore a potential research collaboration concerning diagnostic algorithms (the "Purpose") and may disclose Confidential Information to each other for that Purpose only. 2. Confidential Information. "Confidential Information" means any non-public technical, financial or commercial information disclosed by one party to the other, whether orally, in writing or by inspection, that is marked confidential or would reasonably be understood to be confidential. It excludes information that is publicly available through no fault of the receiving party, was already known to the receiving party, or is independently developed without use of the disclosing party's information. 3. Obligations. Each receiving party shall hold the other's Confidential Information in strict confidence, use it solely for the Purpose, and disclose it only to employees and advisers who need to know it and are bound by written confidentiality obligations no less protective than those in this Agreement. 4. Term. This Agreement remains in force for two (2) years from the Effective Date, and the confidentiality obligations survive for five (5) years after its expiry or termination. 5. Remedies. Each party acknowledges that breach may cause irreparable harm and that the other party may seek injunctive relief in addition to any other remedy. 6. Governing Law. This Agreement is governed by the laws of the State of New York.""",
facts=["Northwind", "Kestrel", "1 March 2026", "New York"]),

"Master Services Agreement": dict(
text="""MASTER SERVICES AGREEMENT. This Master Services Agreement (the "Agreement") is made on 15 January 2026 (the "Effective Date") between Corvane Systems Pvt. Ltd., a company incorporated in India with its registered office in Coimbatore (the "Provider"), and Ardent Logistics Limited, a company incorporated in England and Wales (the "Client"). The Provider has agreed to supply cloud-based fleet management software and related support services to the Client on the terms set out below.

1. Services. The Provider shall deliver the services described in each statement of work signed by both parties (each a "Statement of Work"). Each Statement of Work shall specify the deliverables, milestones, acceptance criteria and applicable fees. In the event of a conflict, the terms of this Agreement prevail over those of a Statement of Work unless the Statement of Work expressly states otherwise.

2. Fees and Payment. The Client shall pay the Provider a fixed annual subscription fee of USD 240,000, invoiced in four equal quarterly instalments of USD 60,000. Each invoice is payable within thirty (30) days of receipt. Overdue amounts bear interest at 1.5% per month from the due date until payment in full. All fees are exclusive of applicable taxes, which the Client shall bear.

3. Term and Renewal. This Agreement commences on the Effective Date and continues for an initial term of three (3) years. It renews automatically for successive one (1) year periods unless either party gives written notice of non-renewal at least ninety (90) days before the end of the then-current term.

4. Service Levels. The Provider shall maintain monthly platform availability of at least 99.9%, excluding scheduled maintenance of which the Client has received at least five (5) business days' notice. If availability falls below 99.0% in any calendar month, the Client shall be entitled to a service credit of ten percent (10%) of that month's pro-rated fee.

5. Data Protection. Each party shall comply with all applicable data protection laws. The Provider shall process personal data only on the Client's documented instructions, implement appropriate technical and organisational security measures, and notify the Client without undue delay and in any event within seventy-two (72) hours of becoming aware of a personal data breach.

6. Intellectual Property. All intellectual property rights in the software and the Provider's pre-existing materials remain with the Provider. The Client retains ownership of all data it supplies and receives a non-exclusive, non-transferable licence to use the software during the term.

7. Limitation of Liability. Neither party excludes liability for death, personal injury caused by negligence, or fraud. Subject to that, each party's total aggregate liability arising under this Agreement in any contract year shall not exceed the fees paid or payable in that contract year, and neither party is liable for indirect or consequential loss.

8. Termination. Either party may terminate this Agreement immediately by written notice if the other commits a material breach that remains unremedied thirty (30) days after written notice requiring remedy, or if the other becomes insolvent. On termination the Provider shall return or delete the Client's data within sixty (60) days.

9. Governing Law and Disputes. This Agreement is governed by the laws of England and Wales. The parties submit to the exclusive jurisdiction of the courts of London for any dispute arising out of or in connection with it.""",
facts=["Corvane", "Ardent", "15 January 2026", "240,000"]),

"Regulatory notice (fictional regulator)": dict(
text="""REGULATORY NOTICE No. NDPRA/2026/14 (sample document, fictional). The National Digital Payments Regulatory Authority (the "Authority") directs all licensed payment service providers to complete migration of customer authentication to the revised two-factor framework by 31 December 2026. Regulated entities must submit a board-approved implementation plan to the Authority's Supervision Department by 30 June 2026 and file quarterly progress reports thereafter. Non-compliance may attract a monetary penalty of up to INR 1,00,00,000 per instance under Section 47A of the Payments Oversight Act, 2019, in addition to supervisory action. Entities with fewer than 50,000 active customer accounts may apply in writing for an extension of up to ninety (90) days, provided the application is received no later than 31 August 2026.""",
facts=["NDPRA/2026/14", "31 December 2026", "30 June 2026", "1,00,00,000"]),
}

summary_rows = []
for name, doc in SAMPLES.items():
    r = summarize_document(doc["text"], model, tokenizer)
    kept = [f for f in doc["facts"] if fact_present(f, r["summary"])]
    summary_rows.append((name, r["source_tokens"], r["summary_tokens"], r["compression_ratio"], len(kept), len(doc["facts"]), len(r["warnings"])))
    print("=" * 100)
    print(f"{name}   |   {r['source_tokens']} -> {r['summary_tokens']} tokens   (ratio {r['compression_ratio']:.2f}, passes: {len(r['levels'])})")
    print("-" * 100)
    print(r["summary"])
    print(f"\nKey facts retained: {len(kept)}/{len(doc['facts'])}   missing: {[f for f in doc['facts'] if f not in kept]}")
    print("Faithfulness warnings:", r["warnings"] if r["warnings"] else "none")

print("\n%-42s %8s %8s %7s %10s %9s" % ("document", "src tok", "sum tok", "ratio", "facts kept", "warnings"))
for n, a, b, c, k, t, w in summary_rows:
    print("%-42s %8d %8d %7.2f %7d/%-2d %9d" % (n, a, b, c, k, t, w))

**Honest limitations to mention in the video.** (1) BillSum is legislative text, not private contracts, so a production system would add contract data (for example from public EDGAR exhibits) and human-written summaries. (2) The faithfulness checker catches *invented* numbers, dates and names, but not a *changed obligation* such as "may" versus "shall", so a human review step remains essential for legal use. (3) Abstractive summaries can still paraphrase defined terms; a hybrid design (extractive pass first, then abstractive) is the natural next improvement.

## 13. Step-by-step dry run (for the explanation video)
This walks **one real document** through every stage so you can narrate it: raw text → preprocessing → sentence split → chunking → tokenisation with positions → encoder → decoder step by step → beam search → post-processing → faithfulness check. A smaller `chunk_tokens` is used so the Master Services Agreement splits into several chunks and the chunking logic is visible.

In [ ]:
def dry_run(doc_text, model, tokenizer, chunk_tokens=400):
    model.eval()
    print("STEP 1 | RAW INPUT (first 300 chars)\n", doc_text[:300].replace("\n", " "), "...\n")

    cleaned = clean_text(doc_text)
    print(f"STEP 2 | PREPROCESSING: whitespace/quotes normalised  ({len(doc_text)} -> {len(cleaned)} characters)\n")

    sents = split_sentences(cleaned)
    print(f"STEP 3 | SENTENCE SPLIT: {len(sents)} sentences; first three:")
    for s in sents[:3]:
        print("   -", s[:110] + ("..." if len(s) > 110 else ""))

    chunks = chunk_document(cleaned, tokenizer, chunk_tokens, overlap_sentences=1)
    print(f"\nSTEP 4 | CHUNKING (max {chunk_tokens} tokens, 1-sentence overlap): {len(chunks)} chunks ->", [n_tokens(c) for c in chunks], "tokens")

    first = chunks[0]
    enc = tokenizer(first, max_length=MAX_SOURCE_TOKENS, truncation=True, return_tensors="pt").to(model.device)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    print("\nSTEP 5 | TOKENISATION of chunk 1 (byte-level BPE; 'Ġ' marks a leading space). First 16 tokens:")
    print("   position:", list(range(16)))
    print("   token   :", toks[:16])
    print("   id      :", enc["input_ids"][0][:16].tolist())
    print("   (each id is mapped to a word embedding, then a learned position embedding for its position is added)")

    with torch.inference_mode():
        enc_out = model.get_encoder()(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    print(f"\nSTEP 6 | ENCODER (12 layers of multi-head self-attention): output shape {tuple(enc_out.last_hidden_state.shape)} = (batch, tokens, d_model)")

    print("\nSTEP 7 | DECODER, first 8 steps (greedy view; each step cross-attends to the encoder output). Top-3 candidates:")
    dec_ids = torch.tensor([[model.config.decoder_start_token_id]], device=model.device)
    for step in range(8):
        with torch.inference_mode():
            out = model(encoder_outputs=enc_out, attention_mask=enc["attention_mask"], decoder_input_ids=dec_ids, use_cache=False)
        probs = torch.softmax(out.logits[0, -1].float(), dim=-1)
        top = torch.topk(probs, 3)
        cands = [(tokenizer.convert_ids_to_tokens(int(i)), round(float(p), 3)) for p, i in zip(top.values, top.indices)]
        print(f"   step {step}: {cands}")
        dec_ids = torch.cat([dec_ids, top.indices[:1].view(1, 1)], dim=1)

    mn, mx = length_budget(n_tokens(first))
    beam = run_generation([first], model, tokenizer, dict(BASE_GEN, min_new_tokens=mn, max_new_tokens=mx), batch_size=1)[0]
    print(f"\nSTEP 8 | BEAM SEARCH (4 beams, no 3-gram repeats, length budget {mn}-{mx} tokens) -> chunk-1 summary:\n  ", beam)

    result = summarize_document(doc_text, model, tokenizer, chunk_tokens=chunk_tokens)
    print(f"\nSTEP 9 | MERGE + POST-PROCESS over all chunks (dedupe, tidy ending). Passes: {result['levels']}")
    print("\nSTEP 10 | FINAL SUMMARY:\n  ", result["summary"])
    print(f"\n   {result['source_tokens']} -> {result['summary_tokens']} tokens (compression ratio {result['compression_ratio']:.2f})")
    print("   Faithfulness warnings:", result["warnings"] if result["warnings"] else "none")
    return result

_ = dry_run(SAMPLES["Master Services Agreement"]["text"], model, tokenizer)

## 14. Save and download the model and results

In [ ]:
os.makedirs(FINAL_DIR, exist_ok=True)
if not DO_FINE_TUNE:                                    # keep the same layout either way
    model.save_pretrained(FINAL_DIR); tokenizer.save_pretrained(FINAL_DIR)

with open("results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)
print("Saved results.json and the model in:", FINAL_DIR)

# Reload test: proves the saved folder works on its own.
reloaded = AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(device)
reloaded_tok = AutoTokenizer.from_pretrained(FINAL_DIR)
print("Reload check ->", summarize_document(SAMPLES["Short notice (single clause)"]["text"], reloaded, reloaded_tok)["summary"])

# Colab only: zip and download (skipped silently elsewhere).
try:
    from google.colab import files
    shutil.make_archive("bart_legal_final", "zip", FINAL_DIR)
    files.download("bart_legal_final.zip")
    files.download("results.json")
except Exception as e:
    print("(download skipped:", type(e).__name__, ")")

## 15. Cheat-sheet for the explanation video
1. **Problem and design choice.** Legal summaries need fluency *and* faithfulness. I chose **abstractive** BART because extractive output is faithful but choppy and cannot compress across clauses, then added a checker to cover the hallucination risk of abstractive models.
2. **Architecture.** Encoder–decoder Transformer; self-attention, 16-head multi-head attention, learned positional embeddings, cross-attention (Section 6 shows each one live).
3. **Transfer learning.** Pretrained BART (`bart-large-cnn`) fine-tuned on BillSum (Section 10), measured against the un-tuned baseline (Section 11).
4. **Pipeline.** clean → sentence split → chunk (overlap) → per-chunk generation with length budget → merge/dedupe → second pass if long → tidy → faithfulness check.
5. **Compression ratio.** About 20% (5:1), capped at about 350 tokens for long documents (Section 3).
6. **Metrics.** ROUGE-1/2/L plus unsupported-number/date/term rates and key-fact retention.
7. **Generalisation.** Unseen California bills plus four hand-written documents of different types and lengths.
8. **Limitations.** Domain gap (statutes vs contracts), checker cannot detect altered obligations, human review still required.
9. **Dry run.** Section 13.